In [1]:
import os
import pandas as pd

from dotenv import load_dotenv
from entsoe import EntsoePandasClient

In [2]:
load_dotenv("../.env")

api_key = os.getenv("ENTSOE_API_KEY")
client = EntsoePandasClient(api_key=api_key)



In [3]:
api_key is not None

True

In [4]:
start = pd.Timestamp("2025-04-01", tz="Europe/Stockholm")
end = pd.Timestamp("2025-04-08", tz="Europe/Stockholm")

load_test = client.query_load_forecast(
    country_code="SE_3", 
    start=start, 
    end=end
)

In [5]:
type(load_test)

pandas.DataFrame

In [6]:
load_test.head()

,Forecasted Load
2025-04-01 00:00:00+02:00,9288.0
2025-04-01 01:00:00+02:00,9118.0
2025-04-01 02:00:00+02:00,9070.0
2025-04-01 03:00:00+02:00,9087.0
2025-04-01 04:00:00+02:00,9191.0


In [7]:
load_test.shape

(168, 1)

In [8]:
load_test.isna().sum()

Forecasted Load    0
dtype: int64

In [9]:
periods = [
    (
        pd.Timestamp("2024-03-08", tz="Europe/Stockholm"),
        pd.Timestamp("2025-01-01", tz="Europe/Stockholm")
    ),
    (
        pd.Timestamp("2025-01-01", tz="Europe/Stockholm"),
        pd.Timestamp("2026-01-01", tz="Europe/Stockholm")
    ),
    (
        pd.Timestamp("2026-01-01", tz="Europe/Stockholm"),
        pd.Timestamp("2026-05-01", tz="Europe/Stockholm")
    )
]

load_parts = []

for start, end in periods:
    part = client.query_load_forecast(
        country_code="SE_3",
        start=start,
        end=end
    )
    
    load_parts.append(part)

In [10]:
load_all = pd.concat(load_parts).sort_index()

In [11]:
load_all = load_all[
    ~load_all.index.duplicated(keep="first")
]

In [12]:
load_all.index = load_all.index.tz_convert("UTC")

In [13]:
load_all = load_all.reset_index()

In [14]:
load_all.head()

,index,Forecasted Load
0,2024-03-07 23:00:00+00:00,10730.0
1,2024-03-08 00:00:00+00:00,10612.0
2,2024-03-08 01:00:00+00:00,10566.0
3,2024-03-08 02:00:00+00:00,10621.0
4,2024-03-08 03:00:00+00:00,10737.0


In [15]:
load_all = load_all.rename(columns={
    "index": "timestamp",
    "Forecasted Load": "load_forecast_mw"
})

In [16]:
start_utc = pd.Timestamp("2024-03-08 00:00", tz="UTC")
end_utc = pd.Timestamp("2026-05-01 00:00", tz="UTC")

load_all = load_all[
    (load_all["timestamp"] >= start_utc) &
    (load_all["timestamp"] < end_utc)
].copy()

In [17]:
load_all["timestamp"].min(), load_all["timestamp"].max()

(Timestamp('2024-03-08 00:00:00+0000', tz='UTC'),
 Timestamp('2026-04-30 21:45:00+0000', tz='UTC'))

In [18]:
load_all.shape

(29515, 2)

In [19]:
load_all["load_forecast_mw"].isna().sum()

np.int64(0)

In [20]:
load_all["timestamp"].duplicated().sum()

np.int64(0)

In [21]:
load_all["timestamp"].diff().value_counts().head(10)

timestamp
0 days 01:00:00    15215
0 days 00:15:00    14298
1 days 00:15:00        1
Name: count, dtype: int64

In [22]:
time_diff = load_all["timestamp"].diff()

load_all.loc[
    time_diff == pd.Timedelta(minutes=15),
    ["timestamp", "load_forecast_mw"]
].head()

,timestamp,load_forecast_mw
15217,2025-12-01 23:15:00+00:00,9498.0
15218,2025-12-01 23:30:00+00:00,9462.0
15219,2025-12-01 23:45:00+00:00,9424.0
15220,2025-12-02 00:00:00+00:00,9379.0
15221,2025-12-02 00:15:00+00:00,9336.0


In [23]:
load_all.loc[
    time_diff > pd.Timedelta(hours=1),
    ["timestamp", "load_forecast_mw"]
]

,timestamp,load_forecast_mw
15408,2025-12-04 23:00:00+00:00,9530.0


In [24]:
gap_positions = load_all.index[
    time_diff > pd.Timedelta(hours=1)
]

for i in gap_positions:
    print("Before:", load_all.loc[i - 1, "timestamp"])
    print("After: ", load_all.loc[i, "timestamp"])

Before: 2025-12-03 22:45:00+00:00
After:  2025-12-04 23:00:00+00:00


In [25]:
time_diff = load_all["timestamp"].diff()

first_15min = load_all.loc[
    time_diff == pd.Timedelta(minutes=15),
    "timestamp"
].iloc[0]

transition = first_15min.floor("h")

transition

Timestamp('2025-12-01 23:00:00+0000', tz='UTC')

In [26]:
time_diff = load_all["timestamp"].diff()

first_15min = load_all.loc[
    time_diff == pd.Timedelta(minutes=15),
    "timestamp"
].iloc[0]

transition = first_15min.floor("h")

transition
gap_start = pd.Timestamp(
    "2025-12-03 20:00",
    tz="UTC"
).tz_convert("Europe/Stockholm")

gap_end = pd.Timestamp(
    "2025-12-05 02:00",
    tz="UTC"
).tz_convert("Europe/Stockholm")

gap_test = client.query_load_forecast(
    country_code="SE_3",
    start=gap_start,
    end=gap_end
)

In [27]:
gap_test.index = gap_test.index.tz_convert("UTC")

In [28]:
gap_test.loc[
    "2025-12-03 22:00":"2025-12-04 23:30"
]

,Forecasted Load
2025-12-03 22:00:00+00:00,10108.0
2025-12-03 22:15:00+00:00,9965.0
2025-12-03 22:30:00+00:00,9859.0
2025-12-03 22:45:00+00:00,9790.0
2025-12-04 23:00:00+00:00,9530.0
2025-12-04 23:15:00+00:00,9530.0
2025-12-04 23:30:00+00:00,9530.0


In [29]:
tail_start = pd.Timestamp(
    "2026-04-30 18:00",
    tz="UTC"
).tz_convert("Europe/Stockholm")

tail_end = pd.Timestamp(
    "2026-05-02 00:00",
    tz="UTC"
).tz_convert("Europe/Stockholm")

tail = client.query_load_forecast(
    country_code="SE_3",
    start=tail_start,
    end=tail_end
)

tail.index = tail.index.tz_convert("UTC")

tail.tail(15)

,Forecasted Load
2026-05-01 20:15:00+00:00,7586.0
2026-05-01 20:30:00+00:00,7519.0
2026-05-01 20:45:00+00:00,7449.0
2026-05-01 21:00:00+00:00,7375.0
2026-05-01 21:15:00+00:00,7321.0
2026-05-01 21:30:00+00:00,7289.0
2026-05-01 21:45:00+00:00,7277.0
2026-05-01 22:00:00+00:00,7238.0
2026-05-01 22:15:00+00:00,7218.0
2026-05-01 22:30:00+00:00,7187.0


In [30]:
tail = tail.reset_index()

tail = tail.rename(columns={
    "index": "timestamp",
    "Forecasted Load": "load_forecast_mw"
})

In [31]:
load_all = pd.concat(
    [
        load_all[["timestamp", "load_forecast_mw"]],
        tail[["timestamp", "load_forecast_mw"]]
    ],
    ignore_index=True
)

In [32]:
load_all = (
    load_all
    .drop_duplicates(subset="timestamp", keep="first")
    .sort_values("timestamp")
    .reset_index(drop=True)
)

In [33]:
tail.loc[
    "2026-04-30 22:00":"2026-05-01 00:00"
]

,timestamp,load_forecast_mw


In [34]:
tail[
    (tail["timestamp"] >= pd.Timestamp("2026-04-30 22:00", tz="UTC")) &
    (tail["timestamp"] <= pd.Timestamp("2026-05-01 00:00", tz="UTC"))
]

,timestamp,load_forecast_mw
16,2026-04-30 22:00:00+00:00,7809.0
17,2026-04-30 22:15:00+00:00,7708.0
18,2026-04-30 22:30:00+00:00,7634.0
19,2026-04-30 22:45:00+00:00,7584.0
20,2026-04-30 23:00:00+00:00,7551.0
21,2026-04-30 23:15:00+00:00,7524.0
22,2026-04-30 23:30:00+00:00,7495.0
23,2026-04-30 23:45:00+00:00,7464.0
24,2026-05-01 00:00:00+00:00,7431.0


In [35]:
load_all = pd.concat(
    [
        load_all[["timestamp", "load_forecast_mw"]],
        tail[["timestamp", "load_forecast_mw"]]
    ],
    ignore_index=True
)

In [36]:
load_all.tail()

,timestamp,load_forecast_mw
29734,2026-05-01 22:45:00+00:00,7145.0
29735,2026-05-01 23:00:00+00:00,7092.0
29736,2026-05-01 23:15:00+00:00,7049.0
29737,2026-05-01 23:30:00+00:00,7017.0
29738,2026-05-01 23:45:00+00:00,6994.0


In [37]:
load_all = (
    load_all
    .drop_duplicates(subset="timestamp", keep="first")
    .sort_values("timestamp")
    .reset_index(drop=True)
)

In [38]:
load_all["timestamp"].duplicated().sum()

np.int64(0)

In [39]:
start_utc = pd.Timestamp("2024-03-08 00:00", tz="UTC")
end_utc = pd.Timestamp("2026-05-01 00:00", tz="UTC")

load_all = load_all[
    (load_all["timestamp"] >= start_utc) &
    (load_all["timestamp"] < end_utc)
].copy()

In [40]:
load_all["timestamp"].min(), load_all["timestamp"].max()

(Timestamp('2024-03-08 00:00:00+0000', tz='UTC'),
 Timestamp('2026-04-30 23:45:00+0000', tz='UTC'))

In [41]:
hourly_load = (
    load_all
    .set_index("timestamp")["load_forecast_mw"]
    .resample("1h")
    .mean()
)

In [42]:
hourly_count = (
    load_all
    .set_index("timestamp")["load_forecast_mw"]
    .resample("1h")
    .count()
)

In [43]:
load_hourly = pd.DataFrame({
    "load_forecast_mw": hourly_load,
    "count": hourly_count
}).reset_index()

In [44]:
load_hourly.head()

,timestamp,load_forecast_mw,count
0,2024-03-08 00:00:00+00:00,10612.0,1
1,2024-03-08 01:00:00+00:00,10566.0,1
2,2024-03-08 02:00:00+00:00,10621.0,1
3,2024-03-08 03:00:00+00:00,10737.0,1
4,2024-03-08 04:00:00+00:00,11201.0,1


In [45]:
load_hourly["count"].value_counts().sort_index()

count
0       24
1    15215
4     3577
Name: count, dtype: int64

In [46]:
transition = pd.Timestamp("2025-12-01 23:00", tz="UTC")

load_hourly["valid"] = (
    (
        (load_hourly["timestamp"] < transition) &
        (load_hourly["count"] == 1)
    )
    |
    (
        (load_hourly["timestamp"] >= transition) &
        (load_hourly["count"] == 4)
    )
)

In [47]:
load_hourly["valid"].value_counts()

valid
True     18792
False       24
Name: count, dtype: int64

In [48]:
load_hourly.loc[
    ~load_hourly["valid"],
    "load_forecast_mw"
] = pd.NA

In [49]:
load_hourly["load_forecast_mw"].isna().sum()

np.int64(24)

In [50]:
load_hourly.to_csv(
    "../data/processed/se3_load_forecast.csv",
    index=False
)

In [51]:
load_hourly.head()

,timestamp,load_forecast_mw,count,valid
0,2024-03-08 00:00:00+00:00,10612.0,1,True
1,2024-03-08 01:00:00+00:00,10566.0,1,True
2,2024-03-08 02:00:00+00:00,10621.0,1,True
3,2024-03-08 03:00:00+00:00,10737.0,1,True
4,2024-03-08 04:00:00+00:00,11201.0,1,True
